# Day 04 上午课堂练习：电商用户数据清洗与预处理

**主数据文件：** E Commerce Dataset.xlsx（使用 E Comm 工作表）

**提交要求：** 完成所有 TODO，运行全部单元后提交本 Notebook 与清洗后的 CSV 文件。

## 学习目标

- 检查字段类型、缺失值和重复记录；
- 使用中位数填补数值缺失；
- 统一类别字段的同义取值；
- 使用统计规则和业务规则检查候选异常值；
- 导出清洗后的数据。

---
## 1. 读取数据

如报找不到文件，请修改候选路径或 DATA_PATH。

In [5]:
from pathlib import Path

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.2f}")

candidates = [
    Path(r"C:\Users\Administrator\Desktop\E Commerce Dataset.xlsx"),
    Path("data/E Commerce Dataset.xlsx"),
]
DATA_PATH = next((path for path in candidates if path.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError("未找到 E Commerce Dataset.xlsx，请修改 DATA_PATH。")

df = pd.read_excel(DATA_PATH, sheet_name="E Comm")
print(f"读取文件：{DATA_PATH}")
print(f"数据形状：{df.shape[0]} 行，{df.shape[1]} 列")
df.head()

读取文件：C:\Users\Administrator\Desktop\E Commerce Dataset.xlsx
数据形状：5630 行，20 列


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,NaN,Phone,1,8.00,UPI,Male,3.00,4,Mobile,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,NaN,Phone,1,30.00,Debit Card,Male,2.00,4,Mobile,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Phone,1,12.00,CC,Male,NaN,3,Mobile,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


### 任务 1：理解数据

运行下一单元，并以注释回答：

1. 一行数据代表什么？
2. 哪个字段是用户唯一标识？
3. 哪个字段可作为用户流失分析的目标变量？

In [6]:
df.info()
# 1. 一行数据代表一位电商平台用户的个人信息、使用习惯、消费相关的完整用户记录
# 2. CustomerID（用户ID）字段是用户唯一标识，每一行的CustomerID都不重复，可以唯一区分每个用户
# 3. Churn（流失标记）字段可作为用户流失分析的目标变量，一般Churn=1代表用户流失、Churn=0代表用户留存，是流失预测任务要预测的标签# 3.

<class 'pandas.DataFrame'>
RangeIndex: 5630 entries, 0 to 5629
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   CustomerID                   5630 non-null   int64  
 1   Churn                        5630 non-null   int64  
 2   Tenure                       5366 non-null   float64
 3   PreferredLoginDevice         5630 non-null   str    
 4   CityTier                     5630 non-null   int64  
 5   WarehouseToHome              5379 non-null   float64
 6   PreferredPaymentMode         5630 non-null   str    
 7   Gender                       5630 non-null   str    
 8   HourSpendOnApp               5375 non-null   float64
 9   NumberOfDeviceRegistered     5630 non-null   int64  
 10  PreferedOrderCat             5630 non-null   str    
 11  SatisfactionScore            5630 non-null   int64  
 12  MaritalStatus                5630 non-null   str    
 13  NumberOfAddress              

---
## 2. 数据质量检查

数据清洗前，先检查缺失值和重复值。

### 任务 2：生成缺失值报告

创建 missing_report，包含“缺失数量”和“缺失比例”两列；按缺失数量降序排列。缺失比例用百分比表示，保留两位小数。

In [10]:
# TODO: 创建 missing_report
# 统计每列缺失数量
missing_count = df.isna().sum()
# 计算缺失比例，转百分比并保留2位小数
missing_ratio = (df.isna().mean() * 100).round(2)

# 拼接成DataFrame
missing_report = pd.DataFrame({
    "缺失数量": missing_count,
    "缺失比例(%)": missing_ratio
})
# 按缺失数量降序排列
missing_report = missing_report.sort_values("缺失数量", ascending=False)

display(missing_report)# display(missing_report)

,缺失数量,缺失比例(%)
DaySinceLastOrder,307,5.45
OrderAmountHikeFromlastYear,265,4.71
Tenure,264,4.69
OrderCount,258,4.58
CouponUsed,256,4.55
HourSpendOnApp,255,4.53
WarehouseToHome,251,4.46
CustomerID,0,0.00
PreferredLoginDevice,0,0.00
Churn,0,0.00


### 任务 3：检查重复记录

分别统计完全重复行数与 CustomerID 重复数量。思考：CustomerID 重复时，能否直接删除？

In [11]:
# TODO: 完成两项重复值统计
# 1. 完全重复行数量
duplicate_rows = df.duplicated().sum()
# 2. CustomerID重复的数量（有重复的用户ID行数）
duplicate_customer_ids = df["CustomerID"].duplicated().sum()

print("完全重复行数：", duplicate_rows)
print("CustomerID重复行数：", duplicate_customer_ids)

# 思考：用户ID重复时，不能直接删除，因为
# 1. 同一CustomerID多条记录可能是同一用户不同时间的行为数据，每条记录业务含义不同，直接删除会丢失时序信息；
# 2. 需先排查重复根源：是数据采集bug、用户多账号登录，还是多条订单/行为记录；
# 3. 若盲目删除会造成样本丢失，影响流失模型训练的数据完整性，应先分组聚合或标记异常再处理。# 思考：用户 ID 重复时，不能直接删除，因为……

完全重复行数： 0
CustomerID重复行数： 0


---
## 3. 缺失值处理

本练习对数值型缺失统一采用中位数填充。缺失不等于 0，例如订单数缺失并不能说明用户没有下单。

### 任务 4：用中位数填补数值缺失

请对下列字段逐列使用中位数填充，随后检查剩余缺失值。

In [12]:
numeric_missing_cols = [
    "Tenure",
    "WarehouseToHome",
    "HourSpendOnApp",
    "OrderAmountHikeFromlastYear",
    "CouponUsed",
    "OrderCount",
    "DaySinceLastOrder",
]

# TODO: 循环填充每列的中位数
for col in numeric_missing_cols:
    # 取该列中位数，填充当前列缺失值
    fill_val = df[col].median()
    df[col] = df[col].fillna(fill_val)

# TODO: 输出上述字段剩余的缺失值数量
print("填充后各数值字段剩余缺失值：")
print(df[numeric_missing_cols].isna().sum())

填充后各数值字段剩余缺失值：
Tenure                         0
WarehouseToHome                0
HourSpendOnApp                 0
OrderAmountHikeFromlastYear    0
CouponUsed                     0
OrderCount                     0
DaySinceLastOrder              0
dtype: int64


### 思考题

什么时候不适合用中位数填充？写出一种场景及更合适的处理思路。

In [ ]:
# 场景：订单数量OrderCount字段缺失代表用户从未下单，中位数填充会凭空赋予用户订单数，扭曲业务逻辑
# 处理思路：使用常量0填充缺失，同时新增标记列is_order_miss记录原始缺失状态，区分真实0单与采集缺失

---
## 4. 类别字段标准化

同一业务含义被写成不同文本，会导致统计结果被拆散。先观察，再统一；不要在没有业务依据的情况下随意合并。

### 任务 5：查看类别取值

检查登录设备、支付方式和订单品类字段，记录你发现的同义类别。

In [15]:
category_cols = [
    "PreferredLoginDevice",
    "PreferredPaymentMode",
    "PreferedOrderCat",
]

for col in category_cols:
    print(f"\n=========={col}==========")
    print(df[col].value_counts())


==========PreferredLoginDevice==========
PreferredLoginDevice
Mobile Phone    2765
Computer        1634
Phone           1231
Name: count, dtype: int64

==========PreferredPaymentMode==========
PreferredPaymentMode
Debit Card          2314
Credit Card         1501
E wallet             614
UPI                  414
COD                  365
CC                   273
Cash on Delivery     149
Name: count, dtype: int64

==========PreferedOrderCat==========
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


### 任务 6：统一同义类别

按以下规则进行标准化：

- Phone → Mobile Phone
- COD → Cash on Delivery
- CC → Credit Card
- Mobile → Mobile Phone

处理后重新输出频数。

In [16]:
# TODO：完成类别标准化
# df["PreferredLoginDevice"] = ...
# df["PreferredPaymentMode"] = ...
# df["PreferedOrderCat"] = ...

# TODO：重新检查类别频数
# for col in category_cols:
#     print(f"\n{col}")
#     print(df[col].value_counts())
# 1.登录设备替换规则：Phone、Mobile统一为 Mobile Phone
df["PreferredLoginDevice"] = df["PreferredLoginDevice"].replace({
    "Phone": "Mobile Phone",
    "Mobile": "Mobile Phone"
})

# 2.支付方式替换：COD→Cash on Delivery，CC→Credit Card
df["PreferredPaymentMode"] = df["PreferredPaymentMode"].replace({
    "COD": "Cash on Delivery",
    "CC": "Credit Card"
})

# 重新打印频数，验证替换结果
for col in category_cols:
    print(f"\n处理后：{col}")
    print(df[col].value_counts())


处理后：PreferredLoginDevice
PreferredLoginDevice
Mobile Phone    3996
Computer        1634
Name: count, dtype: int64

处理后：PreferredPaymentMode
PreferredPaymentMode
Debit Card          2314
Credit Card         1774
E wallet             614
Cash on Delivery     514
UPI                  414
Name: count, dtype: int64

处理后：PreferedOrderCat
PreferedOrderCat
Laptop & Accessory    2050
Mobile Phone          1271
Fashion                826
Mobile                 809
Grocery                410
Others                 264
Name: count, dtype: int64


---
## 5. 候选异常值检查

IQR 方法只能发现候选异常值，不能直接证明数据错误。处理前必须结合业务判断。

In [17]:
def iqr_outlier_summary(series):
    """返回数值字段的 IQR 候选异常值摘要。"""
    series = series.dropna()
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    return pd.Series({
        "Q1": q1,
        "Q3": q3,
        "下限": lower,
        "上限": upper,
        "候选异常值数量": ((series < lower) | (series > upper)).sum()
    })

### 任务 7：检查候选异常值

分别检查 WarehouseToHome、OrderCount 和 CashbackAmount。随后说明：候选异常值能否直接删除，为什么？

In [18]:
# TODO：调用函数检查三个字段
# display(iqr_outlier_summary(df["WarehouseToHome"]))
# display(iqr_outlier_summary(df["OrderCount"]))
# display(iqr_outlier_summary(df["CashbackAmount"]))

# 结论：候选异常值不能直接删除，因为……
def iqr_outlier_summary(series):
    # 计算四分位数
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    # 筛选异常值
    outliers = series[(series < lower_bound) | (series > upper_bound)]
    print(f"Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}")
    print(f"下界:{lower_bound:.2f}, 上界:{upper_bound:.2f}")
    print(f"异常样本数量：{len(outliers)}")
    return outliers

# 调用函数检查三个字段
print("==== WarehouseToHome ====")
iqr_outlier_summary(df["WarehouseToHome"])

print("\n==== OrderCount ====")
iqr_outlier_summary(df["OrderCount"])

print("\n==== CashbackAmount ====")
iqr_outlier_summary(df["CashbackAmount"])

==== WarehouseToHome ====
Q1=9.00, Q3=20.00, IQR=11.00
下界:-7.50, 上界:36.50
异常样本数量：2

==== OrderCount ====
Q1=1.00, Q3=3.00, IQR=2.00
下界:-2.00, 上界:6.00
异常样本数量：703

==== CashbackAmount ====
Q1=145.77, Q3=196.39, IQR=50.62
下界:69.84, 上界:272.33
异常样本数量：438


10     295.45
40     299.26
61     290.33
62     287.22
65     299.99
        ...  
5534   303.75
5537   316.61
5561   321.36
5597   319.31
5603   313.80
Name: CashbackAmount, Length: 438, dtype: float64

### 任务 8：业务规则检查

统计以下不符合业务规则的记录数量：

- 使用时长小于 0；
- 仓库距离小于 0；
- 订单数小于或等于 0；
- 返现金额小于 0。

In [19]:
# TODO：完成业务规则检查
# rules = {
#     "使用时长小于 0": ...,
#     "仓库距离小于 0": ...,
#     "订单数小于或等于 0": ...,
#     "返现金额小于 0": ...,
# }
# pd.Series(rules)
# 构建规则字典
rules = {
    "使用时长小于 0": (df["DaySinceLastOrder"] < 0).sum(),
    "仓库距离小于 0": (df["WarehouseToHome"] < 0).sum(),
    "订单数小于或等于 0": (df["OrderCount"] <= 0).sum(),
    "返现金额小于 0": (df["CashbackAmount"] < 0).sum(),
}

# 转为Series输出查看
import pandas as pd
pd.Series(rules)

使用时长小于 0      0
仓库距离小于 0      0
订单数小于或等于 0    0
返现金额小于 0      0
dtype: int64

---
## 6. 清洗结果验收与导出

在导出前确认：指定数值字段不再有缺失；类别同义值已统一；输出目录存在。

In [24]:
# TODO：完成验收
# assert df[numeric_missing_cols].isna().sum().sum() == 0, "数值字段仍有缺失值"
# assert "Phone" not in df["PreferredLoginDevice"].unique(), "登录设备尚未统一"
# assert "COD" not in df["PreferredPaymentMode"].unique(), "支付方式尚未统一"
# assert "CC" not in df["PreferredPaymentMode"].unique(), "支付方式尚未统一"

# print("数据清洗验收通过。")
# 筛选符合业务规则的数据，生成df_clean
df_clean = df[
    (df["DaySinceLastOrder"] >= 0) &
    (df["WarehouseToHome"] >= 0) &
    (df["OrderCount"] > 0) &
    (df["CashbackAmount"] >= 0)
]

# ----------------下面是验收断言代码----------------
numeric_missing_cols = [
    "WarehouseToHome", "SatisfactionScore", "NumberOfAddress",
    "OrderAmountHikeFromlastYear", "CouponUsed", "OrderCount",
    "DaySinceLastOrder", "CashbackAmount"
]
assert df_clean[numeric_missing_cols].isna().sum().sum() == 0, "数值字段仍有缺失值"
assert "Phone" not in df_clean["PreferredLoginDevice"].unique(), "登录设备尚未统一"
assert "COD" not in df_clean["PreferredPaymentMode"].unique(), "支付方式尚未统一"
assert "CC" not in df_clean["PreferredPaymentMode"].unique(), "支付方式尚未统一"
print("数据清洗验收通过。")

数据清洗验收通过。


### 任务 9：导出清洗后的数据

将文件导出至 output/ecommerce_customer_cleaned.csv。请确保原始数据不会被覆盖。

In [25]:
OUTPUT_PATH = Path("./output/ecommerce_customer_cleaned.csv")
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"已导出: {OUTPUT_PATH.resolve()}")

已导出: C:\Users\Administrator\PyCharmMiscProject\output\ecommerce_customer_cleaned.csv


---
## 7. 提交前自查

- [ ] 已完成缺失值报告；
- [ ] 已完成重复记录检查；
- [ ] 已填补指定数值字段的缺失值；
- [ ] 已统一登录设备、支付方式和订单品类；
- [ ] 已完成候选异常值与业务规则检查；
- [ ] 已导出 ecommerce_customer_cleaned.csv；
- [ ] 已在关键代码处保留注释，说明处理理由。

## 课后思考

若要基于本数据预测用户流失，哪些字段可以作为特征？CustomerID 是否应该用于建模？为什么？